In [1]:
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("Environment:", sys.executable)

Python: 3.12.16 (main, Oct 26 2026, 11:31:43) [Clang 21.1.8 ]
PyTorch: 2.9.1
MPS available: True
Environment: /Users/amelia/anaconda3/envs/cs260d/bin/python


In [2]:
from pathlib import Path

print("Current directory:", Path.cwd())
print("Project source exists:", Path("src").exists())
print("Checkpoints exists:", Path("checkpoints").exists())
print("Parent checkpoints exists:", Path("../checkpoints").exists())

Current directory: /Users/amelia/Desktop/260D final project/CS260D_Final_Project-main
Project source exists: True
Checkpoints exists: True
Parent checkpoints exists: False


In [3]:
from pathlib import Path

checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)

print("Created:", checkpoint_dir.resolve())

Created: /Users/amelia/Desktop/260D final project/CS260D_Final_Project-main/checkpoints


In [4]:
from pathlib import Path

checkpoint_dir = Path("checkpoints")

for file in checkpoint_dir.glob("*.pt"):
    print(file.name, f"{file.stat().st_size / 1024**2:.1f} MB")

print(
    "Dense-Aug ready:",
    (checkpoint_dir / "dense_aug_seed42.pt").is_file()
)

dense_clean_seed42.pt 64.0 MB
dense_clean_seed42_last.pt 127.9 MB
dense_aug_seed42.pt 64.0 MB
dense_aug_seed42_last.pt 127.9 MB
Dense-Aug ready: True


In [5]:
import sys
from pathlib import Path
import torch

# Make the project's src directory importable
project_root = Path.cwd()
sys.path.insert(0, str(project_root / "src"))

# Import the project's pipeline
import occprune.pipeline as pipeline

# Load the pretrained checkpoint
checkpoint_path = project_root / "checkpoints" / "dense_aug_seed42.pt"

checkpoint = pipeline.load_checkpoint(str(checkpoint_path))

print("Checkpoint loaded successfully!")
print("Checkpoint type:", type(checkpoint))

Checkpoint loaded successfully!
Checkpoint type: <class 'tuple'>


In [6]:
print("Number of returned objects:", len(checkpoint))

for i, item in enumerate(checkpoint):
    print(f"\nItem {i}:")
    print("Type:", type(item))

    if isinstance(item, dict):
        print("Keys:", list(item.keys())[:15])
    elif isinstance(item, torch.nn.Module):
        print("Model class:", item.__class__.__name__)
        print("Parameters:", sum(p.numel() for p in item.parameters()))
    else:
        print("Value preview:", str(item)[:300])

Number of returned objects: 3

Item 0:
Type: <class 'torchvision.models.resnet.ResNet'>
Model class: ResNet
Parameters: 11439680

Item 1:
Type: <class 'occprune.faceverif.CosFaceHead'>
Model class: CosFaceHead
Parameters: 5304832

Item 2:
Type: <class 'dict'>
Keys: ['name', 'mode', 'seed', 'epoch', 'val_acc', 'n_classes', 'config']


In [7]:
model, head, metadata = checkpoint

print("Model:", type(model).__name__)
print("Head:", type(head).__name__)
print("Training mode:", metadata.get("mode"))
print("Epoch:", metadata.get("epoch"))
print("Validation accuracy:", metadata.get("val_acc"))

Model: ResNet
Head: CosFaceHead
Training mode: aug
Epoch: 20
Validation accuracy: {'clean': 0.8895, 'square10': 0.8841666666666667, 'square30': 0.8565000000000002, 'square50': 0.7955}


In [8]:
import torch.nn as nn

conv_layers = []

for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):
        conv_layers.append((name, module))

print("Total convolutional layers:", len(conv_layers))

for i, (name, layer) in enumerate(conv_layers):
    print(
        f"{i:2d} | {name:25s} | "
        f"Shape: {tuple(layer.weight.shape)}"
    )

Total convolutional layers: 20
 0 | conv1                     | Shape: (64, 3, 7, 7)
 1 | layer1.0.conv1            | Shape: (64, 64, 3, 3)
 2 | layer1.0.conv2            | Shape: (64, 64, 3, 3)
 3 | layer1.1.conv1            | Shape: (64, 64, 3, 3)
 4 | layer1.1.conv2            | Shape: (64, 64, 3, 3)
 5 | layer2.0.conv1            | Shape: (128, 64, 3, 3)
 6 | layer2.0.conv2            | Shape: (128, 128, 3, 3)
 7 | layer2.0.downsample.0     | Shape: (128, 64, 1, 1)
 8 | layer2.1.conv1            | Shape: (128, 128, 3, 3)
 9 | layer2.1.conv2            | Shape: (128, 128, 3, 3)
10 | layer3.0.conv1            | Shape: (256, 128, 3, 3)
11 | layer3.0.conv2            | Shape: (256, 256, 3, 3)
12 | layer3.0.downsample.0     | Shape: (256, 128, 1, 1)
13 | layer3.1.conv1            | Shape: (256, 256, 3, 3)
14 | layer3.1.conv2            | Shape: (256, 256, 3, 3)
15 | layer4.0.conv1            | Shape: (512, 256, 3, 3)
16 | layer4.0.conv2            | Shape: (512, 512, 3, 3)
17 | layer4.0

In [9]:
import pkgutil
import occprune

print("Available modules:")

for module in pkgutil.iter_modules(occprune.__path__):
    print("-", module.name)

Available modules:
- data
- faceverif
- occlusion
- pipeline
- transforms
- utils


In [10]:
import inspect
import occprune.data as data
import occprune.pipeline as pipeline
import occprune.occlusion as occlusion

for module in [data, pipeline, occlusion]:
    print(f"\n===== {module.__name__} =====")

    for name, obj in inspect.getmembers(module, inspect.isfunction):
        if obj.__module__ == module.__name__:
            print(name, inspect.signature(obj))


===== occprune.data =====
_decode_chunk (args) -> 'int'
_list_webface (zip_path) -> 'tuple[list[str], np.ndarray]'
build_casia_cache (zip_path, cache_dir, workers: 'int' = 12, chunk: 'int' = 4000, verbose: 'bool' = True) -> 'dict'
build_pairs_cache (zip_path, cache_dir, name: 'str') -> 'dict'
casia_paths (cache_dir) -> 'dict'
casia_split_path (seed: 'int')
check_casia_split (split: 'dict', labels: 'np.ndarray') -> 'None'
load_casia (cache_dir) -> 'tuple[np.ndarray, np.ndarray]'
load_casia_split (seed: 'int') -> 'dict'
load_pairs (cache_dir, name: 'str') -> 'dict'
make_casia_split (labels: 'np.ndarray', frac: 'float', min_images: 'int', pairs_per_class: 'int', seed: 'int') -> 'dict'
pairs_cache_path (cache_dir, name: 'str')
save_casia_split (split: 'dict', seed: 'int') -> 'None'
train_indices_and_labels (labels: 'np.ndarray', holdout_ids: 'np.ndarray') -> 'tuple[np.ndarray, np.ndarray, int]'

===== occprune.pipeline =====
_amp_dtype (name: 'str')
_build (cfg, n_cls, dev, seed)
_cfg_see

In [11]:
from pathlib import Path

print("Project root:", Path.cwd())

print("\nCASIA paths:")
print(data.casia_paths(Path("data")))

print("\nTraining config:")
print(metadata["config"].get("data", "No 'data' key"))

Project root: /Users/amelia/Desktop/260D final project/CS260D_Final_Project-main

CASIA paths:
{'images': PosixPath('/Users/amelia/Desktop/260D final project/CS260D_Final_Project-main/data/casia_112.npy'), 'labels': PosixPath('/Users/amelia/Desktop/260D final project/CS260D_Final_Project-main/data/casia_labels.npy'), 'meta': PosixPath('/Users/amelia/Desktop/260D final project/CS260D_Final_Project-main/data/casia_meta.json')}

Training config:
{'webface_zip': 'data/_downloads/webface.zip', 'val_zip': 'data/_downloads/val.zip', 'cache_dir': 'data/cache', 'img_size': 112, 'holdout_frac': 0.02, 'holdout_min_images': 5, 'val_pairs_per_class': 3000, 'test_sets': ['lfw', 'agedb_30', 'calfw', 'cplfw'], 'num_workers': 6}


In [12]:
from pathlib import Path

paths_to_check = [
    "data/_downloads/webface.zip",
    "data/_downloads/val.zip",
    "data/casia_112.npy",
    "data/casia_labels.npy",
    "data/casia_meta.json",
    "data/cache",
]

for path in paths_to_check:
    p = Path(path)
    print(f"{path}: {'EXISTS' if p.exists() else 'MISSING'}")

data/_downloads/webface.zip: EXISTS
data/_downloads/val.zip: EXISTS
data/casia_112.npy: MISSING
data/casia_labels.npy: MISSING
data/casia_meta.json: MISSING
data/cache: EXISTS


In [13]:
from pathlib import Path

download_dir = Path("data/_downloads")
download_dir.mkdir(parents=True, exist_ok=True)

print("Download directory:", download_dir.resolve())
print("Directory exists:", download_dir.exists())

Download directory: /Users/amelia/Desktop/260D final project/CS260D_Final_Project-main/data/_downloads
Directory exists: True


In [14]:
import subprocess

subprocess.run([
    "curl", "-fL", "--retry", "3",
    "-C", "-",
    "-o", "data/_downloads/webface.zip",
    "https://www.kaggle.com/api/v1/datasets/download/yakhyokhuja/webface-112x112"
], check=True)

print("CASIA-WebFace downloaded!")

** Resuming transfer from byte position 2723893755
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


CASIA-WebFace downloaded!


100   173  100   173    0     0    673      0 --:--:-- --:--:-- --:--:--   673


In [15]:
from pathlib import Path
import zipfile

zip_path = Path("data/_downloads/webface.zip")

print("File exists:", zip_path.exists())
print("File size:", round(zip_path.stat().st_size / 1024**3, 2), "GB")

with zipfile.ZipFile(zip_path) as z:
    print("Number of files:", len(z.namelist()))
    print("ZIP structure valid:", z.testzip() is None)

File exists: True
File size: 2.54 GB
Number of files: 490623
ZIP structure valid: True


In [16]:
import subprocess

subprocess.run([
    "curl", "-fL", "--retry", "3",
    "-C", "-",
    "-o", "data/_downloads/val.zip",
    "https://www.kaggle.com/api/v1/datasets/download/yakhyokhuja/agedb-30-calfw-cplfw-lfw-aligned-112x112"
], check=True)

print("Validation datasets downloaded!")

** Resuming transfer from byte position 1408340487
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


Validation datasets downloaded!


100   173  100   173    0     0    650      0 --:--:-- --:--:-- --:--:--   650


In [17]:
from pathlib import Path
import zipfile

val_path = Path("data/_downloads/val.zip")

print("File exists:", val_path.exists())
print("File size:", round(val_path.stat().st_size / 1024**3, 2), "GB")

with zipfile.ZipFile(val_path) as z:
    print("Number of files:", len(z.namelist()))
    print("ZIP structure valid:", z.testzip() is None)

File exists: True
File size: 1.31 GB
Number of files: 48005
ZIP structure valid: True


In [18]:
import shutil
from pathlib import Path

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

total, used, free = shutil.disk_usage(project_root)

print("Project:", project_root)
print(f"Free disk space: {free / 1024**3:.2f} GB")

Project: /Users/amelia/Desktop/260D final project/CS260D_Final_Project-main
Free disk space: 50.39 GB


In [19]:
from pathlib import Path
from occprune.data import load_casia

cache_dir = Path("data/cache")

images, labels = load_casia(cache_dir)

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Images dtype:", images.dtype)

Images shape: (490623, 112, 112, 3)
Labels shape: (490623,)
Images dtype: uint8


In [20]:
from occprune.data import load_casia_split

split = load_casia_split(seed=42)

print("Split type:", type(split))
print("Split keys:", list(split.keys()))

for key, value in split.items():
    if hasattr(value, "shape"):
        print(key, value.shape)
    else:
        print(key, type(value))

Split type: <class 'dict'>
Split keys: ['holdout_ids', 'val_pairs', 'val_issame', 'fold', 'seed']
holdout_ids (211,)
val_pairs (6000, 2)
val_issame (6000,)
fold (6000,)
seed ()


In [21]:
from occprune.data import train_indices_and_labels

train_idx, train_labels, n_classes = train_indices_and_labels(
    labels,
    split["holdout_ids"]
)

print("Training images:", len(train_idx))
print("Training labels:", train_labels.shape)
print("Number of classes:", n_classes)

print("First 5 image indices:", train_idx[:5])
print("First 5 training labels:", train_labels[:5])

Training images: 481210
Training labels: (490623,)
Number of classes: 10361
First 5 image indices: [0 1 2 3 4]
First 5 training labels: [0 0 0 0 0]


In [22]:
import numpy as np

rng = np.random.default_rng(42)

score_indices = rng.choice(
    train_idx,
    size=1024,
    replace=False
)

score_labels = train_labels[score_indices]

print("Selected images:", len(score_indices))
print("Selected labels:", score_labels.shape)
print("Unique identities:", len(np.unique(score_labels)))
print("First 5 indices:", score_indices[:5])
print("First 5 labels:", score_labels[:5])
print("Label range:", score_labels.min(), score_labels.max())

Selected images: 1024
Selected labels: (1024,)
Unique identities: 917
First 5 indices: [271712 334044 477690 241928  88659]
First 5 labels: [4702 6054 9899 4071 1055]
Label range: 15 10360


In [23]:
from pathlib import Path
import numpy as np

score_dir = Path("results/pruning")
score_dir.mkdir(parents=True, exist_ok=True)

np.savez(
    score_dir / "score_samples_seed42.npz",
    indices=score_indices,
    labels=score_labels,
    seed=42
)

print("Saved:", score_dir / "score_samples_seed42.npz")
print("Number of samples:", len(score_indices))

Saved: results/pruning/score_samples_seed42.npz
Number of samples: 1024


In [24]:
import inspect
import occprune.transforms as transforms
import occprune.faceverif as faceverif

for module in [transforms, faceverif]:
    print(f"\n===== {module.__name__} =====")

    for name, obj in inspect.getmembers(module):
        if inspect.isfunction(obj) or inspect.isclass(obj):
            if getattr(obj, "__module__", None) == module.__name__:
                try:
                    print(name, inspect.signature(obj))
                except (TypeError, ValueError):
                    print(name)


===== occprune.transforms =====
boxes_to_mask (boxes: 'np.ndarray | torch.Tensor', H: 'int', W: 'int', device) -> 'torch.Tensor'
preprocess (x_u8: 'torch.Tensor', boxes: 'np.ndarray | None' = None, fill: 'float' = 0.0, flip: 'torch.Tensor | None' = None, img_size: 'int | None' = None) -> 'torch.Tensor'
to_tensor_images (images_u8: 'np.ndarray', device) -> 'torch.Tensor'

===== occprune.faceverif =====
CosFaceHead (in_features: 'int', num_classes: 'int', scale: 'float' = 64.0, margin: 'float' = 0.35)
best_threshold (sims: 'np.ndarray', issame: 'np.ndarray') -> 'tuple[float, float]'
build_face_resnet18 (embedding_dim: 'int' = 512, pretrained: 'bool' = True) -> 'nn.Module'
dataset_seed (base: 'int', name: 'str') -> 'int'
embed (model: 'nn.Module', images: 'np.ndarray', idx: 'np.ndarray | None' = None, rects: 'np.ndarray | None' = None, device=None, batch_size: 'int' = 512, flip: 'bool' = True, fill: 'float' = 0.0, amp: 'str' = 'bf16', channels_last: 'bool' = True) -> 'np.ndarray'
evaluat

In [25]:
import inspect

print("CosFaceHead forward:")
print(inspect.signature(head.forward))

print("\nResNet forward:")
print(inspect.signature(model.forward))

print("\nPreprocess source:")
print(inspect.getsource(transforms.preprocess))

CosFaceHead forward:
(emb: 'torch.Tensor', labels: 'torch.Tensor') -> 'tuple[torch.Tensor, torch.Tensor]'

ResNet forward:
(x: torch.Tensor) -> torch.Tensor

Preprocess source:
def preprocess(
    x_u8: torch.Tensor,
    boxes: np.ndarray | None = None,
    fill: float = 0.0,
    flip: torch.Tensor | None = None,
    img_size: int | None = None,
) -> torch.Tensor:
    """x_u8: (B,3,H,W) uint8 tensor. Returns a normalised float tensor."""
    x = x_u8.float().div_(255.0)
    B, _, H, W = x.shape
    if boxes is not None:
        m = boxes_to_mask(boxes, H, W, x.device)
        x = x.masked_fill(m, fill)
    if flip is not None:
        f = flip.to(x.device).view(-1, 1, 1, 1)
        x = torch.where(f, x.flip(-1), x)
    if img_size is not None and (H != img_size or W != img_size):
        x = F.interpolate(x, size=(img_size, img_size), mode="bilinear", align_corners=False)
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, devic

In [26]:
import inspect
import occprune.pipeline as pipeline

print(inspect.getsource(pipeline._train_step))

def _train_step(model, head, opt, sched, x8, y, boxes, flip, cfg):
    tcfg = cfg["train"]
    dt = _amp_dtype(tcfg["amp"])
    x = preprocess(x8, boxes, cfg["occlusion"]["fill"], flip)
    if tcfg["channels_last"]:
        x = x.contiguous(memory_format=torch.channels_last)
    with torch.autocast(device_type=x.device.type, dtype=dt, enabled=dt is not None):
        emb = model(x)
    logits, cos = head(emb, y)
    loss = F.cross_entropy(logits, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    sched.step()
    return loss.detach(), (cos.detach().argmax(1) == y).sum()



In [27]:
import torch
import torch.nn.functional as F
from occprune.transforms import preprocess, to_tensor_images

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = model.to(device)
head = head.to(device)

model.eval()
head.eval()

# 先选 8 张图片测试
batch_indices = score_indices[:8]

x_np = images[batch_indices]
y_np = train_labels[batch_indices]

x8 = to_tensor_images(x_np, device)
y = torch.as_tensor(y_np, dtype=torch.long, device=device)

# Clean：不添加遮挡，也不随机翻转
x = preprocess(x8, boxes=None, flip=None)

# Forward
emb = model(x)
logits, cos = head(emb, y)
loss = F.cross_entropy(logits, y)

# Backward：只计算梯度，不更新权重
model.zero_grad(set_to_none=True)
head.zero_grad(set_to_none=True)
loss.backward()

print("Device:", device)
print("Batch shape:", tuple(x.shape))
print("Embedding shape:", tuple(emb.shape))
print("Loss:", loss.item())
print("First conv gradient exists:", model.conv1.weight.grad is not None)

Device: mps
Batch shape: (8, 3, 112, 112)
Embedding shape: (8, 512)
Loss: 1.0236103534698486
First conv gradient exists: True


In [28]:
# 先选择 ResNet 中的一个卷积层
layer = model.layer1[0].conv1

# 获取权重和梯度
weights = layer.weight.detach()
gradients = layer.weight.grad.detach()

# 计算 Taylor Importance Scores
scores = (weights * gradients).abs()

print("Weight shape:", weights.shape)
print("Gradient shape:", gradients.shape)
print("Taylor score shape:", scores.shape)

print("Minimum score:", scores.min().item())
print("Maximum score:", scores.max().item())
print("Mean score:", scores.mean().item())

Weight shape: torch.Size([64, 64, 3, 3])
Gradient shape: torch.Size([64, 64, 3, 3])
Taylor score shape: torch.Size([64, 64, 3, 3])
Minimum score: 0.0
Maximum score: 0.09915971010923386
Mean score: 0.0024632359854876995


In [29]:
import torch
import torch.nn.functional as F

target_layer = model.layer1[0].conv1

batch_size = 16
total_samples = 0

gradient_sum = torch.zeros_like(target_layer.weight)

model.eval()
head.eval()

for start in range(0, len(score_indices), batch_size):

    batch_indices = score_indices[start:start + batch_size]

    x_np = images[batch_indices]
    y_np = train_labels[batch_indices]

    x8 = to_tensor_images(x_np, device)
    y = torch.as_tensor(y_np, dtype=torch.long, device=device)

    x = preprocess(x8, boxes=None, flip=None)

    model.zero_grad(set_to_none=True)
    head.zero_grad(set_to_none=True)

    emb = model(x)
    logits, cos = head(emb, y)

    loss = F.cross_entropy(logits, y)
    loss.backward()

    n = len(batch_indices)
    gradient_sum += target_layer.weight.grad.detach() * n
    total_samples += n

gradient_avg = gradient_sum / total_samples

clean_taylor_scores = (
    target_layer.weight.detach() * gradient_avg
).abs()

print("Total samples:", total_samples)
print("Score shape:", clean_taylor_scores.shape)
print("Mean Taylor score:", clean_taylor_scores.mean().item())
print("Maximum Taylor score:", clean_taylor_scores.max().item())

Total samples: 1024
Score shape: torch.Size([64, 64, 3, 3])
Mean Taylor score: 0.0002388438006164506
Maximum Taylor score: 0.021878665313124657


In [30]:
import torch.nn as nn

conv_layers = [
    (name, module)
    for name, module in model.named_modules()
    if isinstance(module, nn.Conv2d)
]

print("Total Conv2d layers:", len(conv_layers))

for i, (name, layer) in enumerate(conv_layers):
    print(f"{i:2d}. {name:35s} {tuple(layer.weight.shape)}")

Total Conv2d layers: 20
 0. conv1                               (64, 3, 7, 7)
 1. layer1.0.conv1                      (64, 64, 3, 3)
 2. layer1.0.conv2                      (64, 64, 3, 3)
 3. layer1.1.conv1                      (64, 64, 3, 3)
 4. layer1.1.conv2                      (64, 64, 3, 3)
 5. layer2.0.conv1                      (128, 64, 3, 3)
 6. layer2.0.conv2                      (128, 128, 3, 3)
 7. layer2.0.downsample.0               (128, 64, 1, 1)
 8. layer2.1.conv1                      (128, 128, 3, 3)
 9. layer2.1.conv2                      (128, 128, 3, 3)
10. layer3.0.conv1                      (256, 128, 3, 3)
11. layer3.0.conv2                      (256, 256, 3, 3)
12. layer3.0.downsample.0               (256, 128, 1, 1)
13. layer3.1.conv1                      (256, 256, 3, 3)
14. layer3.1.conv2                      (256, 256, 3, 3)
15. layer4.0.conv1                      (512, 256, 3, 3)
16. layer4.0.conv2                      (512, 512, 3, 3)
17. layer4.0.downsam

In [31]:
# 排除最开始的 conv1，保留其余 19 层
target_layers = {
    name: layer
    for name, layer in conv_layers
    if name != "conv1"
}

print("Number of target layers:", len(target_layers))
print("Total target weights:",
      sum(layer.weight.numel() for layer in target_layers.values()))

Number of target layers: 19
Total target weights: 11157504


In [32]:
# 为 19 个卷积层分别创建梯度累加器
gradient_sums = {
    name: torch.zeros_like(layer.weight)
    for name, layer in target_layers.items()
}

batch_size = 16
total_samples = 0

model.eval()
head.eval()

for start in range(0, len(score_indices), batch_size):

    batch_indices = score_indices[start:start + batch_size]

    x_np = images[batch_indices]
    y_np = train_labels[batch_indices]

    x8 = to_tensor_images(x_np, device)
    y = torch.as_tensor(y_np, dtype=torch.long, device=device)

    # Clean-Taylor：不添加遮挡
    x = preprocess(x8, boxes=None, flip=None)

    model.zero_grad(set_to_none=True)
    head.zero_grad(set_to_none=True)

    emb = model(x)
    logits, cos = head(emb, y)

    loss = F.cross_entropy(logits, y)
    loss.backward()

    n = len(batch_indices)

    # 同时累积 19 层的梯度
    with torch.no_grad():
        for name, layer in target_layers.items():
            gradient_sums[name] += layer.weight.grad * n

    total_samples += n

# 计算每个权重的 Taylor Importance Score
clean_scores = {}

with torch.no_grad():
    for name, layer in target_layers.items():
        avg_grad = gradient_sums[name] / total_samples
        clean_scores[name] = (
            layer.weight * avg_grad
        ).abs().cpu()

# 保存结果
save_path = score_dir / "clean_taylor_scores_seed42.pt"

torch.save({
    "scores": clean_scores,
    "num_samples": total_samples,
    "seed": 42,
    "method": "clean_taylor",
    "score_definition": "abs(weight * mean_gradient)",
    "excluded_layers": ["conv1"],
}, save_path)

print("Total samples:", total_samples)
print("Scored layers:", len(clean_scores))
print("Total scored weights:",
      sum(s.numel() for s in clean_scores.values()))
print("Saved to:", save_path)

Total samples: 1024
Scored layers: 19
Total scored weights: 11157504
Saved to: results/pruning/clean_taylor_scores_seed42.pt


In [33]:
import inspect
import occprune.occlusion as occlusion

print(inspect.getsource(occlusion.sample_train_boxes))

def sample_train_boxes(
    n: int,
    prob: float,
    ratio_range: tuple[float, float],
    H: int,
    W: int,
    rng: np.random.Generator,
) -> np.ndarray:
    """Training-time occlusion: with prob `prob` an image gets a square whose
    area ratio ~ U(ratio_range); otherwise side = 0 (clean image)."""
    lo, hi = ratio_range
    apply = rng.random(n) < prob
    ratios = rng.uniform(lo, hi, size=n)
    sides = np.minimum(np.rint(np.sqrt(ratios * H * W)), min(H, W)).astype(np.int64)
    sides = np.where(apply, sides, 0)
    return _place(sides, H, W, rng)



In [34]:
from pathlib import Path
import yaml

with open(Path("configs/default.yaml"), "r") as f:
    cfg = yaml.safe_load(f)

print("Occlusion configuration:")
print(cfg["occlusion"])

Occlusion configuration:
{'fill': 0.0, 'train_prob': 0.5, 'train_ratio': [0.1, 0.3], 'val_ratios': [0.1, 0.3, 0.5], 'test_ratios': [0.1, 0.3, 0.5], 'test_regions': ['eyes', 'mouth_nose', 'upper_half', 'lower_half'], 'val_seed': 1001, 'test_seed': 2002, 'diag_seed': 3003}


In [35]:
from occprune.occlusion import sample_train_boxes
import numpy as np

# 使用独立随机种子，保证遮挡可复现
mixed_rng = np.random.default_rng(42)

mixed_boxes = sample_train_boxes(
    n=len(score_indices),
    prob=cfg["occlusion"]["train_prob"],
    ratio_range=tuple(cfg["occlusion"]["train_ratio"]),
    H=112,
    W=112,
    rng=mixed_rng
)

print("Boxes shape:", mixed_boxes.shape)
print("First 5 boxes:")
print(mixed_boxes[:5])

Boxes shape: (1024, 3)
First 5 boxes:
[[ 16  92   0]
 [ 22  49  45]
 [111  98   0]
 [  1 102   0]
 [ 25  19  56]]


In [36]:
# 第三列是遮挡正方形的边长
clean_count = np.sum(mixed_boxes[:, 2] == 0)
occluded_count = np.sum(mixed_boxes[:, 2] > 0)

print("Total images:", len(mixed_boxes))
print("Clean images:", clean_count)
print("Occluded images:", occluded_count)
print("Occlusion percentage:", occluded_count / len(mixed_boxes) * 100)

# 保存遮挡框
mixed_boxes_path = score_dir / "mixed_boxes_seed42.npy"
np.save(mixed_boxes_path, mixed_boxes)

print("Saved to:", mixed_boxes_path)

Total images: 1024
Clean images: 509
Occluded images: 515
Occlusion percentage: 50.29296875
Saved to: results/pruning/mixed_boxes_seed42.npy


In [37]:
# 取前 8 张图片
batch_indices = score_indices[:8]
batch_boxes = mixed_boxes[:8]

x_np = images[batch_indices]
y_np = train_labels[batch_indices]

x8 = to_tensor_images(x_np, device)
y = torch.as_tensor(y_np, dtype=torch.long, device=device)

# Mixed-Taylor：使用预先生成的遮挡框
x = preprocess(
    x8,
    boxes=batch_boxes,
    fill=cfg["occlusion"]["fill"],
    flip=None
)

model.zero_grad(set_to_none=True)
head.zero_grad(set_to_none=True)

# Forward
emb = model(x)
logits, cos = head(emb, y)
loss = F.cross_entropy(logits, y)

# Backward
loss.backward()

print("Batch shape:", tuple(x.shape))
print("Occluded images in batch:", int(np.sum(batch_boxes[:, 2] > 0)))
print("Loss:", loss.item())
print(
    "Gradient exists:",
    model.layer1[0].conv1.weight.grad is not None
)

Batch shape: (8, 3, 112, 112)
Occluded images in batch: 2
Loss: 1.8741238117218018
Gradient exists: True


In [38]:
# 为 19 个卷积层创建梯度累加器
mixed_gradient_sums = {
    name: torch.zeros_like(layer.weight)
    for name, layer in target_layers.items()
}

batch_size = 16
total_samples = 0

model.eval()
head.eval()

for start in range(0, len(score_indices), batch_size):

    batch_indices = score_indices[start:start + batch_size]
    batch_boxes = mixed_boxes[start:start + batch_size]

    x_np = images[batch_indices]
    y_np = train_labels[batch_indices]

    x8 = to_tensor_images(x_np, device)
    y = torch.as_tensor(y_np, dtype=torch.long, device=device)

    # Mixed：使用固定的 Clean + Occluded 遮挡框
    x = preprocess(
        x8,
        boxes=batch_boxes,
        fill=cfg["occlusion"]["fill"],
        flip=None
    )

    model.zero_grad(set_to_none=True)
    head.zero_grad(set_to_none=True)

    emb = model(x)
    logits, cos = head(emb, y)

    loss = F.cross_entropy(logits, y)
    loss.backward()

    n = len(batch_indices)

    # 累积所有 19 层的梯度
    with torch.no_grad():
        for name, layer in target_layers.items():
            mixed_gradient_sums[name] += layer.weight.grad * n

    total_samples += n

# 计算 Mixed-Taylor Scores
mixed_scores = {}

with torch.no_grad():
    for name, layer in target_layers.items():
        avg_grad = mixed_gradient_sums[name] / total_samples

        mixed_scores[name] = (
            layer.weight * avg_grad
        ).abs().cpu()

# 保存结果
mixed_save_path = score_dir / "mixed_taylor_scores_seed42.pt"

torch.save({
    "scores": mixed_scores,
    "num_samples": total_samples,
    "seed": 42,
    "method": "mixed_taylor",
    "score_definition": "abs(weight * mean_gradient)",
    "excluded_layers": ["conv1"],
    "occlusion_prob": cfg["occlusion"]["train_prob"],
    "occlusion_ratio": cfg["occlusion"]["train_ratio"],
}, mixed_save_path)

print("Total samples:", total_samples)
print("Scored layers:", len(mixed_scores))
print("Total scored weights:",
      sum(s.numel() for s in mixed_scores.values()))
print("Saved to:", mixed_save_path)

Total samples: 1024
Scored layers: 19
Total scored weights: 11157504
Saved to: results/pruning/mixed_taylor_scores_seed42.pt


In [39]:
# 测试：逐张图片计算 Taylor Scores
test_layer = model.layer1[0].conv1

score_sum = torch.zeros_like(test_layer.weight)

model.eval()
head.eval()

for idx in score_indices[:8]:
    x_np = images[[int(idx)]]
    y_np = train_labels[[int(idx)]]

    x8 = to_tensor_images(x_np, device)
    y = torch.as_tensor(y_np, dtype=torch.long, device=device)

    # Clean：不添加遮挡
    x = preprocess(x8, boxes=None, flip=None)

    model.zero_grad(set_to_none=True)
    head.zero_grad(set_to_none=True)

    emb = model(x)
    logits, cos = head(emb, y)

    loss = F.cross_entropy(logits, y)
    loss.backward()

    # 关键：每张图片先取绝对值，再累加
    with torch.no_grad():
        score_sum += (
            test_layer.weight * test_layer.weight.grad
        ).abs()

test_scores = score_sum / 8

print("Processed images:", 8)
print("Score shape:", test_scores.shape)
print("Mean score:", test_scores.mean().item())
print("Max score:", test_scores.max().item())
print("All finite:", torch.isfinite(test_scores).all().item())

Processed images: 8
Score shape: torch.Size([64, 64, 3, 3])
Mean score: 0.004544886760413647
Max score: 0.22850359976291656
All finite: True


In [41]:
# Seed 42: Clean-Taylor (per-sample scoring)

clean_score_sums = {
    name: torch.zeros_like(layer.weight)
    for name, layer in target_layers.items()
}

model.eval()
head.eval()

for i, idx in enumerate(score_indices):

    x_np = images[[int(idx)]]
    y_np = train_labels[[int(idx)]]

    x8 = to_tensor_images(x_np, device)
    y = torch.as_tensor(y_np, dtype=torch.long, device=device)

    x = preprocess(x8, boxes=None, flip=None)

    model.zero_grad(set_to_none=True)
    head.zero_grad(set_to_none=True)

    emb = model(x)
    logits, cos = head(emb, y)

    loss = F.cross_entropy(logits, y)
    loss.backward()

    with torch.no_grad():
        for name, layer in target_layers.items():
            clean_score_sums[name] += (
                layer.weight * layer.weight.grad
            ).abs()

    if (i + 1) % 128 == 0:
        print(f"Processed {i + 1}/1024 images")

# 计算平均 Taylor Score，并移到 CPU
clean_scores_v2 = {
    name: (score_sum / len(score_indices)).cpu()
    for name, score_sum in clean_score_sums.items()
}

# 保存新版本，避免覆盖旧结果
clean_v2_path = score_dir / "clean_taylor_per_sample_seed42.pt"

torch.save({
    "scores": clean_scores_v2,
    "num_samples": len(score_indices),
    "seed": 42,
    "method": "clean_taylor",
    "score_definition": "mean(abs(weight * per_sample_gradient))",
    "excluded_layers": ["conv1"],
}, clean_v2_path)

print("Total samples:", len(score_indices))
print("Scored layers:", len(clean_scores_v2))
print("Total scored weights:",
      sum(s.numel() for s in clean_scores_v2.values()))
print("Saved to:", clean_v2_path)

Processed 128/1024 images
Processed 256/1024 images
Processed 384/1024 images
Processed 512/1024 images
Processed 640/1024 images
Processed 768/1024 images
Processed 896/1024 images
Processed 1024/1024 images
Total samples: 1024
Scored layers: 19
Total scored weights: 11157504
Saved to: results/pruning/clean_taylor_per_sample_seed42.pt


In [42]:
mixed_rng = np.random.default_rng(42)

# 随机选出 512 张需要遮挡的图片
occluded_positions = mixed_rng.choice(
    len(score_indices),
    size=512,
    replace=False
)

mixed_boxes_exact = sample_train_boxes(
    n=len(score_indices),
    prob=1.0,
    ratio_range=(0.1, 0.3),
    H=112,
    W=112,
    rng=mixed_rng
)

# 其余 512 张保持无遮挡
clean_mask = np.ones(len(score_indices), dtype=bool)
clean_mask[occluded_positions] = False
mixed_boxes_exact[clean_mask, 2] = 0

np.save(
    score_dir / "mixed_boxes_exact_seed42.npy",
    mixed_boxes_exact
)

print("Clean:", np.sum(mixed_boxes_exact[:, 2] == 0))
print("Occluded:", np.sum(mixed_boxes_exact[:, 2] > 0))

Clean: 512
Occluded: 512


In [43]:
# Seed 42: Mixed-Taylor (per-sample scoring)

mixed_score_sums = {
    name: torch.zeros_like(layer.weight)
    for name, layer in target_layers.items()
}

model.eval()
head.eval()

for i, idx in enumerate(score_indices):

    x_np = images[[int(idx)]]
    y_np = train_labels[[int(idx)]]

    x8 = to_tensor_images(x_np, device)
    y = torch.as_tensor(y_np, dtype=torch.long, device=device)

    # 与 Clean-Taylor 的区别：应用固定遮挡框
    boxes = mixed_boxes_exact[i:i+1]

    x = preprocess(
        x8,
        boxes=boxes,
        fill=cfg["occlusion"]["fill"],
        flip=None
    )

    model.zero_grad(set_to_none=True)
    head.zero_grad(set_to_none=True)

    emb = model(x)
    logits, cos = head(emb, y)

    loss = F.cross_entropy(logits, y)
    loss.backward()

    # 每张图片先计算 |weight × gradient|，再累加
    with torch.no_grad():
        for name, layer in target_layers.items():
            mixed_score_sums[name] += (
                layer.weight * layer.weight.grad
            ).abs()

    if (i + 1) % 128 == 0:
        print(f"Processed {i + 1}/1024 images")

# 求平均，得到最终 Mixed-Taylor Scores
mixed_scores_v2 = {
    name: (score_sum / len(score_indices)).cpu()
    for name, score_sum in mixed_score_sums.items()
}

# 保存为新文件，不覆盖旧版本
mixed_v2_path = score_dir / "mixed_taylor_per_sample_seed42.pt"

torch.save({
    "scores": mixed_scores_v2,
    "num_samples": len(score_indices),
    "seed": 42,
    "method": "mixed_taylor",
    "score_definition": "mean(abs(weight * per_sample_gradient))",
    "clean_count": 512,
    "occluded_count": 512,
    "occlusion_ratio": [0.1, 0.3],
    "excluded_layers": ["conv1"],
}, mixed_v2_path)

print("Total samples:", len(score_indices))
print("Scored layers:", len(mixed_scores_v2))
print("Total scored weights:",
      sum(s.numel() for s in mixed_scores_v2.values()))
print("Saved to:", mixed_v2_path)

Processed 128/1024 images
Processed 256/1024 images
Processed 384/1024 images
Processed 512/1024 images
Processed 640/1024 images
Processed 768/1024 images
Processed 896/1024 images
Processed 1024/1024 images
Total samples: 1024
Scored layers: 19
Total scored weights: 11157504
Saved to: results/pruning/mixed_taylor_per_sample_seed42.pt


In [44]:
# Seed 43: select 1024 training images
rng43 = np.random.default_rng(43)

score_indices_43 = rng43.choice(
    train_idx,
    size=1024,
    replace=False
)

score_labels_43 = train_labels[score_indices_43]

# Save sample indices
np.savez(
    score_dir / "score_samples_seed43.npz",
    indices=score_indices_43,
    labels=score_labels_43,
    seed=43
)

print("Seed:", 43)
print("Selected images:", len(score_indices_43))
print("Unique identities:", len(np.unique(score_labels_43)))
print("Saved to:", score_dir / "score_samples_seed43.npz")

Seed: 43
Selected images: 1024
Unique identities: 903
Saved to: results/pruning/score_samples_seed43.npz


In [45]:
def compute_clean_taylor(indices, seed):
    score_sums = {
        name: torch.zeros_like(layer.weight)
        for name, layer in target_layers.items()
    }

    model.eval()
    head.eval()

    for i, idx in enumerate(indices):
        x_np = images[[int(idx)]]
        y_np = train_labels[[int(idx)]]

        x8 = to_tensor_images(x_np, device)
        y = torch.as_tensor(y_np, dtype=torch.long, device=device)

        x = preprocess(x8, boxes=None, flip=None)

        model.zero_grad(set_to_none=True)
        head.zero_grad(set_to_none=True)

        emb = model(x)
        logits, cos = head(emb, y)
        loss = F.cross_entropy(logits, y)
        loss.backward()

        with torch.no_grad():
            for name, layer in target_layers.items():
                score_sums[name] += (
                    layer.weight * layer.weight.grad
                ).abs()

        if (i + 1) % 128 == 0:
            print(f"Seed {seed}: {i + 1}/{len(indices)}")

    scores = {
        name: (value / len(indices)).cpu()
        for name, value in score_sums.items()
    }

    save_path = score_dir / f"clean_taylor_per_sample_seed{seed}.pt"

    torch.save({
        "scores": scores,
        "num_samples": len(indices),
        "seed": seed,
        "method": "clean_taylor",
        "score_definition": "mean(abs(weight * per_sample_gradient))",
        "excluded_layers": ["conv1"],
    }, save_path)

    print("Saved to:", save_path)
    return scores

In [46]:
clean_scores_43 = compute_clean_taylor(
    score_indices_43,
    seed=43
)

Seed 43: 128/1024
Seed 43: 256/1024
Seed 43: 384/1024
Seed 43: 512/1024
Seed 43: 640/1024
Seed 43: 768/1024
Seed 43: 896/1024
Seed 43: 1024/1024
Saved to: results/pruning/clean_taylor_per_sample_seed43.pt


In [50]:
rng43 = np.random.default_rng(43)

# 随机选出 512 张需要遮挡的图片
occluded_positions_43 = rng43.choice(
    len(score_indices_43),
    size=512,
    replace=False
)

# 生成面积为 10%–30% 的随机方块遮挡
mixed_boxes_43 = sample_train_boxes(
    n=len(score_indices_43),
    prob=1.0,
    ratio_range=tuple(cfg["occlusion"]["train_ratio"]),
    H=112,
    W=112,
    rng=rng43
)

# 其余 512 张保持 Clean
clean_mask_43 = np.ones(len(score_indices_43), dtype=bool)
clean_mask_43[occluded_positions_43] = False
mixed_boxes_43[clean_mask_43, 2] = 0

# 保存遮挡框
boxes_path_43 = score_dir / "mixed_boxes_exact_seed43.npy"
np.save(boxes_path_43, mixed_boxes_43)

print("Clean:", np.sum(mixed_boxes_43[:, 2] == 0))
print("Occluded:", np.sum(mixed_boxes_43[:, 2] > 0))
print("Saved to:", boxes_path_43)

Clean: 512
Occluded: 512
Saved to: results/pruning/mixed_boxes_exact_seed43.npy


In [51]:
def compute_mixed_taylor(indices, boxes, seed):
    assert len(indices) == len(boxes)

    score_sums = {
        name: torch.zeros_like(layer.weight)
        for name, layer in target_layers.items()
    }

    model.eval()
    head.eval()

    for i, idx in enumerate(indices):
        x_np = images[[int(idx)]]
        y_np = train_labels[[int(idx)]]

        x8 = to_tensor_images(x_np, device)
        y = torch.as_tensor(y_np, dtype=torch.long, device=device)

        x = preprocess(
            x8,
            boxes=boxes[i:i+1],
            fill=cfg["occlusion"]["fill"],
            flip=None
        )

        model.zero_grad(set_to_none=True)
        head.zero_grad(set_to_none=True)

        emb = model(x)
        logits, cos = head(emb, y)
        loss = F.cross_entropy(logits, y)
        loss.backward()

        # 每张图片先计算绝对 Taylor Score，再累加
        with torch.no_grad():
            for name, layer in target_layers.items():
                score_sums[name] += (
                    layer.weight * layer.weight.grad
                ).abs()

        if (i + 1) % 128 == 0:
            print(f"Seed {seed}: {i + 1}/{len(indices)}")

    scores = {
        name: (value / len(indices)).cpu()
        for name, value in score_sums.items()
    }

    save_path = score_dir / f"mixed_taylor_per_sample_seed{seed}.pt"

    torch.save({
        "scores": scores,
        "num_samples": len(indices),
        "seed": seed,
        "method": "mixed_taylor",
        "score_definition": "mean(abs(weight * per_sample_gradient))",
        "clean_count": int(np.sum(boxes[:, 2] == 0)),
        "occluded_count": int(np.sum(boxes[:, 2] > 0)),
        "occlusion_ratio": list(cfg["occlusion"]["train_ratio"]),
        "excluded_layers": ["conv1"],
    }, save_path)

    print("Saved to:", save_path)
    return scores

In [52]:
mixed_scores_43 = compute_mixed_taylor(
    score_indices_43,
    mixed_boxes_43,
    seed=43
)

Seed 43: 128/1024
Seed 43: 256/1024
Seed 43: 384/1024
Seed 43: 512/1024
Seed 43: 640/1024
Seed 43: 768/1024
Seed 43: 896/1024
Seed 43: 1024/1024
Saved to: results/pruning/mixed_taylor_per_sample_seed43.pt


In [53]:
# Seed 44: select 1024 training images
rng44 = np.random.default_rng(44)

score_indices_44 = rng44.choice(
    train_idx,
    size=1024,
    replace=False
)

score_labels_44 = train_labels[score_indices_44]

# 保存样本索引
np.savez(
    score_dir / "score_samples_seed44.npz",
    indices=score_indices_44,
    labels=score_labels_44,
    seed=44
)

print("Seed:", 44)
print("Selected images:", len(score_indices_44))
print("Unique identities:", len(np.unique(score_labels_44)))
print("Saved to:", score_dir / "score_samples_seed44.npz")

Seed: 44
Selected images: 1024
Unique identities: 922
Saved to: results/pruning/score_samples_seed44.npz


In [54]:
clean_scores_44 = compute_clean_taylor(
    score_indices_44,
    seed=44
)

Seed 44: 128/1024
Seed 44: 256/1024
Seed 44: 384/1024
Seed 44: 512/1024
Seed 44: 640/1024
Seed 44: 768/1024
Seed 44: 896/1024
Seed 44: 1024/1024
Saved to: results/pruning/clean_taylor_per_sample_seed44.pt


In [56]:
rng44 = np.random.default_rng(44)

occluded_positions_44 = rng44.choice(
    len(score_indices_44),
    size=512,
    replace=False
)

mixed_boxes_44 = sample_train_boxes(
    n=len(score_indices_44),
    prob=1.0,
    ratio_range=(0.1, 0.3),
    H=112,
    W=112,
    rng=rng44
)

clean_mask_44 = np.ones(len(score_indices_44), dtype=bool)
clean_mask_44[occluded_positions_44] = False

mixed_boxes_44[clean_mask_44, 2] = 0

np.save(
    score_dir / "mixed_boxes_exact_seed44.npy",
    mixed_boxes_44
)

print("Clean:", np.sum(mixed_boxes_44[:, 2] == 0))
print("Occluded:", np.sum(mixed_boxes_44[:, 2] > 0))

Clean: 512
Occluded: 512


In [57]:
mixed_scores_44 = compute_mixed_taylor(
    score_indices_44,
    mixed_boxes_44,
    seed=44
)

Seed 44: 128/1024
Seed 44: 256/1024
Seed 44: 384/1024
Seed 44: 512/1024
Seed 44: 640/1024
Seed 44: 768/1024
Seed 44: 896/1024
Seed 44: 1024/1024
Saved to: results/pruning/mixed_taylor_per_sample_seed44.pt
